# Data Wrangling 2.4 Solutions

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2

import json

import csv

from datetime import datetime as dt

from IPython.display import display, HTML


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

## You try it - join stage_3_sales to stage_3_line items to show the details for the contradictions on total_amount

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

with a as (
    select sa.store_id,
           sa.sale_id
    from stage_3_sales as sa
    where total_amount::numeric <> (select sum(quantity::numeric) * 12 
                                from stage_3_line_items as l 
                                where sa.store_id = l.store_id and sa.sale_id = l.sale_id)
    )

select sa.stage_id,
       sa.store_id,
       sa.sale_id,
       sa.customer_id,
       sa.sale_date,
       sa.total_amount,
       l.line_item_id,
       l.product_id,
       l.quantity
from stage_3_sales as sa
     join stage_3_line_items as l
         on sa.store_id = l.store_id and sa.sale_id = l.sale_id
where (sa.store_id, sa.sale_id) in (select * from a)
order by stage_id, store_id, sale_id, line_item_id


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - find line items without a sale

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

select *
from stage_3_line_items as l
where (l.store_id, l.sale_id) not in (select store_id, sale_id from stage_3_sales)
order by stage_id, store_id, sale_id

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - check for uniformity in capitalization in last names in stage_3_customers

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

select cu1.*
from stage_3_customers cu1
     join customers cu2
         on cu1.customer_id::numeric = cu2.customer_id
where cu1.last_name <> cu2.last_name
      and lower(cu1.last_name) = lower(cu1.last_name)

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)